file: ALL.wgs.mergedSV.v8.20130502.svs.genotypes.vcf.gz  
from: https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/phase3/integrated_sv_map/

file: ALL.chr22.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz  
from: https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/


Unlike in the example from the nootebook, this snp file has 255 info lines at the beginning, including the one with column names:  
``bcftools view -h <file> | wc -l``

In [24]:
import pandas as pd

In [ ]:
import pandas as pd 
import gzip 

root_dir = 'data/'
file = 'ALL.chr22.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz'
new_data_header = ""
# get header
with gzip.open(f"{root_dir}/{file}", 'rt') as f_in:
    for line in f_in:
        if not line.startswith('##'):
            new_data_header = line
            break

print(new_data_header)

# load data
# load genotype
genotypes = pd.read_csv(
  f'{root_dir}/{file}', 
  comment='#', 
  sep='\t', 
  names=new_data_header.strip().split('\t'), 
  header=None,
  compression='gzip'
).iloc[:, 9:]
genotypes = genotypes.T
headers = genotypes.columns[:]
genotypes

In [ ]:
ped_file = f'{root_dir}/integrated_call_samples.20130502.ALL.ped'
pedigree = pd.read_csv(ped_file, sep='\t', index_col='Individual ID')
pedigree

In [ ]:
Y_train = pedigree.loc[genotypes.index]['Population']
display(Y_train.shape)
display(Y_train.head())

In [ ]:
import allel
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import squareform

## LD Analysis — First 500 SNPs

In [ ]:
LD_N_SNPS = 500

callset     = allel.read_vcf(f'{root_dir}/{file}', fields=['calldata/GT', 'samples'], stop=LD_N_SNPS)
vcf_samples = list(callset['samples'])

# Filter to pedigreed samples (Y_train.index)
valid_vcf_idx = [vcf_samples.index(s) for s in Y_train.index if s in vcf_samples]

g  = allel.GenotypeArray(callset['calldata/GT'])
gn = g[:, valid_vcf_idx].to_n_alt(fill=-1)

r  = allel.rogers_huff_r(gn)
LD = squareform(r ** 2)
print(f'LD matrix shape: {LD.shape}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im = axes[0].imshow(LD, cmap='hot_r', vmin=0, vmax=1)
axes[0].set_title(f'LD matrix (r²) — first {LD_N_SNPS} SNPs, Chr22')
axes[0].set_xlabel('SNP index')
axes[0].set_ylabel('SNP index')
plt.colorbar(im, ax=axes[0])

bins   = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
ld_max = np.amax(LD, axis=1)
axes[1].hist(ld_max, bins=bins, edgecolor='black')
axes[1].set_title(f'Distribution of max r² per SNP')
axes[1].set_xlabel('max r²')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

bin_labels = np.digitize(ld_max, bins=bins, right=True)
labels, counts = np.unique(bin_labels, return_counts=True)
for l, c in zip(labels, counts):
    print(f'r² bin {bins[l-1]:.1f}–{bins[l]:.1f}: {c} SNPs')

## STICI Training

In [20]:
import importlib.util
import sys
from types import SimpleNamespace

# STICI_V1.1.py has dots in the filename so standard import does not work
_spec = importlib.util.spec_from_file_location('STICI', 'STICI_V1.1.py')
_mod  = importlib.util.module_from_spec(_spec)
sys.modules['STICI'] = _mod
_spec.loader.exec_module(_mod)

from STICI import DataReader, train_the_model

Tensorflow version 2.21.0


In [ ]:
ARGS = SimpleNamespace(
    ref                = f'{root_dir}/{file}',
    save_dir           = f'{root_dir}/training_results/snp',
    tihp               = True,
    mode               = 'train',
    min_mr             = 0.85,
    max_mr             = 0.95,
    cs                 = 2048,
    sites_per_model    = 10240,
    co                 = 64,
    na_heads           = 16,
    embed_dim          = 128,
    batch_size_per_gpu = 4,
    lr                 = 0.002,
    use_r2             = True,
    epochs             = 1000,
    val_n_batches      = 8,
    random_seed        = 2022,
    restart_training   = False,
    which_chunk        = -1,
    ref_vac            = False,
    ref_sep            = None,
    ref_file_format    = 'infer',
    ref_fcai           = False,
    ref_comment        = '##',
    verbose            = 1,
)

In [23]:
train_the_model(ARGS)

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)
Num gpus to be used: 1
Ref file format is vcf.
Reading the file...


KeyboardInterrupt: 